# 02 — Huấn luyện & Đánh giá
Notebook này cho phép bạn train và đánh giá mô hình trực tiếp trong Jupyter.

In [ ]:
import sys; sys.path.insert(0, '..')
import tensorflow as tf
print(f'TF: {tf.__version__}')

## 1. Tải dữ liệu

In [ ]:
from utils.dataset import create_generators, get_class_weights
train_gen, val_gen, test_gen = create_generators(augment=True)
cw = get_class_weights(train_gen)
print('Class weights:', cw)

## 2. Xây dựng mô hình

In [ ]:
from utils.model import build_model
model, base_model = build_model('mobilenetv2')

## 3. Giai đoạn 1: Frozen base

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cbs = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

hist1 = model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=cbs, class_weight=cw, verbose=1)

## 4. Giai đoạn 2: Fine-tuning

In [ ]:
from utils.model import unfreeze_model
unfreeze_model(base_model, n_layers=30)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist2 = model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=cbs, class_weight=cw, verbose=1)

## 5. Đánh giá & Lưu đồ thị

In [ ]:
from utils.evaluate import evaluate_model, plot_confusion_matrix, plot_history
import json

result = evaluate_model(model, test_gen, verbose=True)
plot_confusion_matrix(result['y_true'], result['y_pred'], save=True, show=True)

full_hist = {k: hist1.history.get(k,[]) + hist2.history.get(k,[]) for k in ['loss','accuracy','val_loss','val_accuracy']}
plot_history(full_hist, save=True, show=True)

## 6. Lưu mô hình

In [ ]:
from config import BEST_MODEL_PATH
model.save(str(BEST_MODEL_PATH))
print(f'Mô hình lưu tại: {BEST_MODEL_PATH}')